# Softmax
*Converting raw scores into probabilities — the universal normalizer of deep learning.*

---
**Component:** `learning/kernels/softmax/`
**Companion kernels:** `v0_naive.py` → `sm89_v1_block_per_row.py` → `sm89_v2_smem_tree_reduce.py` → `sm89_v3_online_single_pass.py`


## What Is Softmax?

**In plain English:** Softmax takes a list of arbitrary numbers ("scores" or "logits") and converts them into a probability distribution — a list of non-negative numbers that sum to exactly 1.

Think of it as a **voting machine**: each score gets exponentially more votes the higher it is, then all votes are divided by the total so you end up with percentages.

**Where it appears in a transformer:**
- **Attention weights** — "how much does token A pay attention to token B?"
- **Output head** — "what is the probability of each next word in the vocabulary?"


In [ ]:
import math

# We'll trace through everything by hand, then compare to library versions
print("Setup complete — numpy and math imported")


## Worked Example: Step by Step

**Input (logits):**

$$x = [2.0,\ 1.0,\ 0.1]$$

| Step | Operation | Result |
|------|-----------|--------|
| 1 | $e^{2.0}$ | 7.389 |
| 1 | $e^{1.0}$ | 2.718 |
| 1 | $e^{0.1}$ | 1.105 |
| 2 | Sum: $7.389 + 2.718 + 1.105$ | 11.212 |
| 3 | $7.389 / 11.212$ | **0.659** |
| 3 | $2.718 / 11.212$ | **0.242** |
| 3 | $1.105 / 11.212$ | **0.099** |

**Output:** $p = [0.659,\ 0.242,\ 0.099]$

✅ Check: $0.659 + 0.242 + 0.099 = 1.000$
✅ Highest input (2.0) → highest probability (0.659)
✅ All values in (0, 1)


In [ ]:
x = [2.0, 1.0, 0.1]

print("Step 1 — exponentiate each element:")
exps = [math.exp(xi) for xi in x]
for xi, ei in zip(x, exps):
    print(f"  exp({xi}) = {ei:.3f}")

total = sum(exps)
print(f"\nStep 2 — sum: {' + '.join(f'{e:.3f}' for e in exps)} = {total:.3f}")

p = [e / total for e in exps]
print(f"\nStep 3 — divide each by {total:.3f}:")
for ei, pi in zip(exps, p):
    print(f"  {ei:.3f} / {total:.3f} = {pi:.3f}")

print(f"\nOutput: {[round(pi, 3) for pi in p]}")
print(f"Sum check: {sum(p):.10f}")


## The Formula

$$\boxed{\text{softmax}(x)_i = \frac{e^{x_i}}{\displaystyle\sum_{j=1}^{n} e^{x_j}}}$$

### Symbol Definitions

| Symbol | Name | Meaning in our example |
|--------|------|----------------------|
| $x_i$ | logit at index $i$ | Raw score, e.g. $x_0 = 2.0$ |
| $e$ | Euler's number | $\approx 2.718$; makes outputs positive |
| $e^{x_i}$ | exponential of $x_i$ | Always $> 0$; grows fast with $x_i$ |
| $n$ | length of vector | $n = 3$ in our example |
| $\sum_{j=1}^{n}$ | sum over all $j$ | Ensures the denominator = total votes |
| $\text{softmax}(x)_i$ | output probability $i$ | $\in (0,1)$, all outputs sum to 1 |

### 🗣️ Say It Out Loud
> *"The softmax of vector x at position i equals: e to the power of x-sub-i, divided by the sum — for all j from 1 to n — of e to the power of x-sub-j."*

**Tutor's gloss:** "We're turning raw scores into a competition. The `exp` makes every score positive and amplifies differences — a score of 2 gets $e^2 = 7.4$ votes while a score of 1 gets only $e^1 = 2.7$ votes. Dividing by the total converts absolute vote counts into percentages."


## The Stability Problem — and the Fix

### Why Naive Softmax Breaks

Try $x = [1000.0,\ 1001.0]$:

$$e^{1000} \approx 5 \times 10^{434} \quad \text{(float32 max is } \approx 3.4 \times 10^{38}\text{)}$$

Result: `inf / inf = NaN` — training crashes.

### The Stability Fix: Subtract the Maximum

Let $m = \max_j x_j$. Then:

$$\text{softmax}(x)_i = \frac{e^{x_i - m}}{\displaystyle\sum_j e^{x_j - m}}$$

### 🗣️ Why This Is Mathematically Identical

$$\frac{e^{x_i - m}}{\sum_j e^{x_j - m}} = \frac{e^{x_i} \cdot e^{-m}}{\sum_j e^{x_j} \cdot e^{-m}} = \frac{e^{x_i}}{\sum_j e^{x_j}}$$

The $e^{-m}$ **cancels top and bottom** — the result is unchanged.
After shifting, every exponent $\leq 0$, so $e^{x_i - m} \in (0, 1]$. No overflow possible.


In [ ]:
def softmax_naive(x):
    exps = [math.exp(xi) for xi in x]
    total = sum(exps)
    return [e / total for e in exps]

def softmax_stable(x):
    m = max(x)
    exps = [math.exp(xi - m) for xi in x]
    total = sum(exps)
    return [e / total for e in exps]

# Overflow test
x_large = [1000.0, 1001.0]
print("Naive on [1000, 1001]:")
try:
    result = softmax_naive(x_large)
    nan_inf = any(not (0 < v < 1) for v in result)
    print(f"  {result}  {'← BAD (NaN/inf)' if nan_inf else 'ok'}")
except (OverflowError, ZeroDivisionError) as e:
    print(f"  CRASHED: {e}")

print("\nStable on [1000, 1001]:")
result = softmax_stable(x_large)
print(f"  {[round(v, 4) for v in result]}  ← correct!")
assert abs(sum(result) - 1.0) < 1e-9


## Online Softmax: Computing Percentages Without Seeing All the Scores First

**The problem with two passes:**
Standard stable softmax needs two scans of the data: one to find the maximum, one to compute `exp` values and sum them. Two scans means the GPU reads every element twice from HBM. On a memory-bound operation like softmax, that's a 33% tax you'd rather not pay.

**Can you compute percentages without knowing the total first?**

Imagine you're a sports commentator reporting finishing times as runners cross the line, one at a time. You need to express each runner's time as a *share of the total* — but you don't know how many runners there are or what the fastest time will be until the last runner finishes.

The trick: keep a running total that's always expressed *relative to the best time seen so far*. When a new record is set, you don't throw out your running total — you adjust it with one multiplication, then carry on. At the end you have everything you need without having stored every individual time.

**The running state — just two numbers:**

| Variable | What it tracks |
|---|---|
| `m` | The largest element seen so far |
| `d` | The sum `Σ exp(xⱼ − m)` for all elements seen so far |

**Update when a new element `xᵢ` arrives:**

$$m_\text{new} = \max(m_\text{old},\ x_i)$$

$$d_\text{new} = d_\text{old} \cdot \underbrace{e^{m_\text{old} - m_\text{new}}}_{\text{rescale old sum}} + \underbrace{e^{x_i - m_\text{new}}}_{\text{add new term}}$$

**Why does `exp(m_old − m_new)` fix things?**

Every term in `d_old` has the form `exp(xⱼ − m_old)`. Now the max is higher. Each of those terms *should* be `exp(xⱼ − m_new)` to stay on a consistent baseline. The conversion is:

```
exp(xⱼ − m_old) × exp(m_old − m_new) = exp(xⱼ − m_new)
```

Instead of going back and recomputing every previous term individually, multiply the *entire old sum* by `exp(m_old − m_new)` in one step. All terms are corrected simultaneously.

**When no new max arrives** (`xᵢ ≤ m_old`): `m_new = m_old`, so `exp(m_old − m_new) = exp(0) = 1`. The old sum is unchanged. Just add the new term. This is the typical case — rescaling costs nothing.

**The invariant that makes it correct** — true after processing any prefix of elements:
- `m` = exact maximum of all elements seen so far
- `d` = exactly `Σ exp(xⱼ − m)` for those elements

At the end, output the softmax as normal: `softmax(x)ᵢ = exp(xᵢ − m) / d`. Same result as two-pass — computed in one scan.

### Why this matters beyond standalone softmax

This two-number trick is what makes **FlashAttention** possible. Standard attention produces a T×T score matrix: at T=4096 that's 64 MB per head, written to HBM and read back before softmax can run. With online softmax, you never need to store those scores: maintain `(m, d)` in registers as you stream through KV tiles, and write only the final output. The T×T matrix never exists anywhere.

In [ ]:
def softmax_online(x):
    m = float('-inf')
    d = 0.0
    for xi in x:
        m_new = max(m, xi)
        rescale = math.exp(m - m_new) if m != float('-inf') else 0.0
        d = d * rescale + math.exp(xi - m_new)
        m = m_new
    return [math.exp(xi - m) / d for xi in x]

# Trace through x = [2.0, 1.0, 0.1]
x = [2.0, 1.0, 0.1]
m, d = float('-inf'), 0.0
print(f"Start:  m = -∞,  d = 0.0\n")
for xi in x:
    m_new = max(m, xi)
    rescale = math.exp(m - m_new) if m != float('-inf') else 0.0
    d_new = d * rescale + math.exp(xi - m_new)
    print(f"See {xi:4.1f}:  m_new = {m_new:.1f},  "
          f"rescale = {rescale:.3f},  d_new = {d_new:.3f}")
    m, d = m_new, d_new

result = [math.exp(xi - m) / d for xi in x]
print(f"\nFinal: m={m}, d={d:.3f}")
print(f"Output: {[round(v, 3) for v in result]}")
assert all(abs(a - b) < 1e-9 for a, b in zip(result, softmax_stable(x)))
print("✓ Matches stable 2-pass result")


## Parallel Merge Operator (for GPU Tree Reduction)

When 128 threads each process a chunk of the row, every thread produces its own local state $(m_t, d_t)$. To combine them into one global $(m, d)$:

$$\text{merge}\bigl((m_1, d_1),\ (m_2, d_2)\bigr) = \Bigl(\max(m_1, m_2),\ d_1 \cdot e^{m_1 - m} + d_2 \cdot e^{m_2 - m}\Bigr)$$

where $m = \max(m_1, m_2)$.

This operator is **associative**: `merge(merge(A,B), C) == merge(A, merge(B,C))`
→ a binary tree works; $\log_2(128) = 7$ rounds suffice for 128 threads.

**This is the same operation used in FlashAttention** to merge partial attention outputs computed on separate KV tiles.

### Log-Sum-Exp Connection

$$\text{LSE}(x) = \log\!\sum_j e^{x_j} = m + \log(d)$$

FlashAttention stores LSE per row (not the full softmax) so the backward pass can recompute $e^{x_i - \text{LSE}} = \text{softmax}(x)_i$ without materializing the $T \times T$ attention matrix.


## Optimization Ladder

| Version | Threads/row | HBM passes | Key idea |
|---------|-------------|------------|----------|
| `v0_naive` | 1 | 3 | One thread does everything serially |
| `sm89_v1_block_per_row` | 128 | 3 | 128 threads split columns; tree reduce in SMEM |
| `sm89_v2_smem_tree_reduce` | 128 | 3 | Better SMEM layout; explicit tree reduction |
| `sm89_v3_online_single_pass` | 128 | **2** | Online fuses pass-1+pass-2; 33% less HBM reads |

**Why memory bandwidth dominates:**
Softmax does ≈5 FLOPs per element but reads/writes 8 bytes.
Arithmetic intensity ≈ 0.6 FLOP/byte vs RTX 4080 ridge point ≈ 230 FLOP/byte.
→ Memory is the bottleneck, not compute. Reducing passes = the win.


## CuTeDSL: Online Softmax in One Pass

The v3 kernel (`sm89_v3_online_single_pass`) fuses the max-finding and sum-finding passes
into a single scan. Each thread tracks a tiny running state — just two numbers — instead of
storing intermediate values.

```python
@cute.kernel
def softmax_online_kernel(mIn, mOut):
    row = blockIdx.x    # one CTA per row
    tid = threadIdx.x   # 0 to 127

    # Each thread tracks: the max it has seen, and the exp-sum relative to that max
    m_local = float('-inf')
    d_local = 0.0

    # One pass over the row — each element is read exactly once
    for col in cutlass.range(tid, N, 128):
        xi = mIn[row, col]

        m_new = max(m_local, xi)
        # When the max rises, all previous exp() terms need rescaling.
        # exp(xi - m_old) = exp(xi - m_new) * exp(m_old - m_new)
        # So multiply the old sum by exp(m_old - m_new), then add the new term.
        d_local = d_local * cute.exp(m_local - m_new) + cute.exp(xi - m_new)
        m_local = m_new

    # Merge 128 local (m, d) states into one global (m, d) via shared memory
    smem_m[tid] = m_local
    smem_d[tid] = d_local
    cute.sync_threads()

    for half in [64, 32, 16, 8, 4, 2, 1]:
        if tid < half:
            m1, d1 = smem_m[tid],        smem_d[tid]
            m2, d2 = smem_m[tid + half], smem_d[tid + half]
            m_new = max(m1, m2)
            smem_m[tid] = m_new
            smem_d[tid] = d1 * cute.exp(m1 - m_new) + d2 * cute.exp(m2 - m_new)
        cute.sync_threads()

    global_m = smem_m[0]
    global_d = smem_d[0]

    # Second pass — can't avoid this, but it's the only second pass
    for col in cutlass.range(tid, N, 128):
        mOut[row, col] = cute.exp(mIn[row, col] - global_m) / global_d
```

**The rescaling line unpacked:**
`d_local = d_local * exp(m_local - m_new) + exp(xi - m_new)`

Think of `d_local` as a sum that was correct when the running max was `m_local`.
A new element `xi` arrives with a higher value `m_new`.
Every previous exp term was computed as `exp(something - m_local)`.
To stay consistent, each must become `exp(something - m_new)`.
That's the same as multiplying each by `exp(m_local - m_new)`.
So we multiply the whole old sum by that factor, then add the new term.
No values are stored — only `m_local` and `d_local` live in registers.

## RTX 4080: Why Passes Matter More Than FLOPs

Softmax does ~5 FLOPs per element against 4 bytes (2 read + 2 write) in BF16.
Intensity: ~1.25 FLOP/byte — still well below the 151 F/B ridge. **Memory-bound.**

The v0 naive kernel makes 3 HBM passes (find max, compute sum, normalize).
The v3 online kernel makes 2 passes (fused scan + normalize pass).
That 33% reduction in HBM reads translates directly to 33% faster kernel time.

**No further architectural tricks help here** — you can't do softmax in one total pass because you need the final (m, d) before you can write any output. Two is the minimum.

For attention specifically: the benefit compounds with FlashAttention. Flash uses online softmax so it never writes the full [T × T] score matrix to HBM. For T=4096: saving 4096 × 4096 × 4 bytes = 64 MB per head per layer from being written and re-read.

## Warp Shuffle: Better Reduction for Short Attention Rows

The v2 kernel uses a shared-memory tree reduce. That's appropriate for long rows
(e.g., vocabulary softmax with N=150,000). But attention softmax rows are short —
N is the sequence length, typically 128 to 4096 — and warp shuffles are faster.

**At N=128:** 128 elements ÷ 128 threads = 1 element per thread.
The entire reduction fits inside one warp shuffle. Zero SMEM, five rounds.

**At N=4096:** 4096 ÷ 128 threads = 32 elements per thread.
Each of the 4 warps runs its own 5-round shuffle reduce.
Then only the 4 warp-level results go into SMEM for one final pass.
Total SMEM traffic: 4 writes + 4 reads, vs 128 writes + 128 reads in the tree approach.

**Warp-shuffle merge for two (m, d) states:**
```python
for offset in [16, 8, 4, 2, 1]:
    m_other = cute.shfl_xor(m_lane, offset)   # get the paired lane's max
    d_other = cute.shfl_xor(d_lane, offset)   # get the paired lane's denominator
    m_new   = max(m_lane, m_other)
    d_lane  = d_lane * exp(m_lane - m_new) + d_other * exp(m_other - m_new)
    m_lane  = m_new
# After 5 rounds: lane 0 holds the warp's global (m, d)
```

**Why faster:** Each shuffle instruction takes ~4 cycles and touches no memory.
SMEM loads have ~20+ cycle latency. At N=4096, the hybrid approach trades
128 SMEM accesses for 4, saving ~120 × 20 = 2,400 cycles of latency per softmax row.

## Where Softmax Lives in the Qwen3-8B Decode Step

Softmax appears in two distinct places per decode pass:

```
Place 1 — Inside attention (step ⑨, ×36 layers, ×32 heads):
  Input:  scores [T] — one score per context position per Q-head
  Output: weights[T] — probability distribution over T positions
  Kernel: NOT a standalone call — fused inside FlashAttention's KV tile loop
  Size at T=4096: 4096 × 32 × 36 × 4 bytes = 18 MB of softmax state (but never written to HBM)

Place 2 — Output head (step 39, once per decode step):
  Input:  logits [151,936] — one logit per vocabulary token
  Output: probabilities or just top-K for sampling
  Kernel: standalone, but rarely the full 151K-element softmax
          top-p sampling typically softmaxes only the top 50–1000 logits
```

### Why attention softmax must run online

In FlashAttention, the attention kernel never writes the full score vector `[T]` to HBM. All T scores don't fit in registers simultaneously, and writing them would defeat the purpose. So softmax is computed incrementally using the online algorithm:

```
For each KV tile (loaded into SMEM, never touches HBM again):
  For each position s in the tile:
    score = Q · K[s] / sqrt(d)
    m_new = max(m_running, score)
    d_running = d_running × exp(m_running - m_new) + exp(score - m_new)
    acc      = acc      × exp(m_running - m_new) + exp(score - m_new) × V[s]
    m_running = m_new

final_out = acc / d_running   ← the only HBM write
```

The two numbers `(m_running, d_running)` live in registers throughout the entire loop. No score is stored. The softmax result is implicit in the accumulator.

### The two-number state captures everything

After seeing all T scores:
```
m = the maximum score seen
d = sum of exp(score - m) for all scores seen
```
Any output element can be recovered: `p[i] = exp(scores[i] - m) / d`.

This is why FlashAttention's backward pass can recompute attention weights without storing the `[T, T]` matrix — it only stores `LSE = m + log(d)`, a `[B, H, T]` tensor, and recomputes everything else on the fly.

## Softmax Implementations in the Wild

### FlashAttention (fused — no standalone kernel)

Inside `flash_attn` CUDA kernels, softmax is never a separate operation.
The `(m, d)` running state is maintained in thread-local registers across the KV tile loop.
At the end of each tile, the inter-warp merge formula combines partial states.
FlashAttention stores log-sum-exp (LSE = m + log(d)) in a `[B, H, T]` tensor for the backward pass, which allows the backward to recompute attention weights without storing the full `[T, T]` matrix.

### PyTorch `F.softmax`

CPU and CUDA implementations that call into cuDNN or custom CUDA kernels.
Not fused — writes intermediate exp values to HBM, reads them back for normalization.
Appropriate for the vocabulary softmax (run once per step) and for debugging.

### Online softmax paper (Milakov & Gimelshein, 2018)

The mathematical foundation: shows that two-pass softmax can be replaced by a single scan maintaining `(max, exp-sum)` running state. This is what every FlashAttention variant uses for attention and what `sm89_v3_online_single_pass` implements for standalone softmax.

### Triton softmax tutorial

`triton/python/tutorials/02-fused-softmax.py` — the canonical introduction to Triton.
Implements two-pass fused softmax (load row once, compute exp + sum, normalize).
Achieves ~95% of cuDNN softmax throughput in ~50 lines.
Does **not** implement true online single-pass — the row still fits in SMEM and is computed in one block, so two passes over registers rather than two passes over HBM.

### Pattern across all implementations

Every production-quality softmax does three things:
1. **Numerically stable** — subtract running max before exp
2. **Two-pass minimum** — first pass gets `(m, d)`, second pass normalizes (online fuses these into one HBM scan + one normalize scan = 2 HBM passes, not 3)
3. **Memory-bound aware** — warp shuffles replace SMEM tree reduces for short rows (attention), saving hundreds of SMEM accesses per row

## One-Sentence Takeaway

Softmax's core challenge is numerical stability (large inputs overflow float32 before normalization can happen), and the max-subtraction trick solves it exactly without changing the result — but the deeper insight is that the entire two-pass algorithm can be fused into one scan by tracking only two running numbers (the max seen and the exp-sum relative to that max), which is precisely what allows FlashAttention to compute attention scores, softmax, and value aggregation in a single pass through the KV cache without ever writing the T×T attention probability matrix to HBM, making the choice between SMEM tree reduction and warp-shuffle reduction a secondary concern compared to the fundamental question of how many times the input row is read from global memory.

## Community Optimization Resources for Softmax on SM89

### Softmax is always fused — standalone kernels are for learning only

In production Qwen3-8B inference, standalone softmax kernels never appear in the attention hot path. The online softmax algorithm is fused inside FlashAttention, where it processes each KV tile without ever writing scores to HBM. The only standalone softmax that runs is the vocabulary softmax at the output head (N=151,936 — one call per token, not 36 per layer).

| Use case | Kernel | Notes |
|---|---|---|
| Attention weights | Fused inside FlashAttention tile loop | Never materialized as T×T matrix |
| Vocabulary output (lm_head) | Standalone softmax or top-K only | N=151,936, once per decode step |
| Training (cross-entropy) | Fused with log in `log_softmax` | Not relevant for inference |

---

### Speedup reference: standalone softmax benchmarks

| Optimization | Hardware | Before | After | Speedup | Implementation |
|---|---|---|---|---|---|
| Fused softmax (1 HBM pass) | RTX 4090 (SM89) | 3 passes naive | 2 passes fused | **~1.5×** | Triton tutorial |
| Online softmax (2 passes) | RTX 4090 (SM89) | 3 passes | 2 passes | **~1.3×** | Triton / CUDA |
| Warp shuffle vs SMEM tree | RTX 3090 (SM86) | 7 barriers (SMEM) | 0 barriers (shuffle) | **~1.2×** short rows | CuTeDSL / CUDA |

**The biggest win (T×T elimination) is 4096× fewer HBM bytes at T=4096 — that's the FlashAttention fusion, not a standalone kernel optimization.**

---

### Warp shuffle reduction for attention softmax rows (N ≤ 512)

At short attention context lengths, the entire row fits inside one warp:

```python
# N=128 (short context or one warp's worth):
# Each thread holds elements [tid, tid+32, tid+64, tid+96]
m_lane = max(my_4_elements)
d_lane = sum(exp(xi - m_lane) for xi in my_4_elements)

# 5-round warp butterfly merge — 0 SMEM, 0 __syncthreads():
for offset in [16, 8, 4, 2, 1]:
    m_other = cute.shfl_xor(m_lane, offset)
    d_other = cute.shfl_xor(d_lane, offset)
    m_new   = max(m_lane, m_other)
    d_lane  = d_lane * exp(m_lane - m_new) + d_other * exp(m_other - m_new)
    m_lane  = m_new
# Lane 0 now has global (m, d) — broadcast back with shfl_broadcast
```

This is 32× fewer SMEM accesses than the 7-barrier tree reduce. At decode (N=1–2048 context tokens per query), this is the right reduction strategy.

---

### Key papers and references

- **Online softmax (Milakov & Gimelshein, 2018):** the mathematical foundation for the 2-pass online algorithm; all FlashAttention variants cite this.
- **FlashAttention (Dao et al., 2022):** fuses online softmax with KV tile processing; eliminates T×T matrix.
- **FlashAttention-2 (Dao, 2023):** parallelizes over Q dimension for better decode occupancy.
- **Triton softmax tutorial:** `openai/triton` `tutorials/02-fused-softmax.py` — canonical introduction to fused 2-pass softmax in ~50 lines.
- **Anatomy of Triton Attention (Luo, 2024):** step-by-step guide to writing a FlashAttention-style kernel in Triton, with explicit online softmax merge.

---

### What to optimize for Qwen3-8B

**Vocabulary softmax (N=151,936, once per decode step):**
- Use top-K shortcut: run softmax only over the top 50–1000 logits for sampling
- Full softmax is ~2 µs at N=151,936 on RTX 4080 — not a bottleneck
- Triton `tl.softmax` in-kernel is sufficient

**Attention softmax (fused in FlashAttention):**
- The online merge (`m_lane`, `d_lane`) runs inside the FlashAttention KV tile loop
- Warp shuffle is preferred over SMEM tree for the inter-warp merge at decode
- SM89 `shfl.sync.bfly.b32` PTX: 4-cycle latency, 0 SMEM — the right tool for short rows